# induction_3 — does induction form at scale?

GPT-2-small (12L / 768d / 12H, block 512), BPE vocab 8192, trained on a 247.5M-token OpenWebText subset (`scripts/transformer_vast.py`, run on a Vast 4090). Same induction-score + heatmap tools as `interp_1` / `Interp_2`.

The arc so far:
- **induction_1** (char, small): induction ≈ 0.017 despite the model copying novel names — blamed character-level tokenization.
- **induction_2** (BPE, *same* small scale): still ≈ 0.013 — tokenization **refuted**.
- **induction_3** (this): if induction lifts off the floor here, it was **scale + data** all along, not tokenization.

Result: best-val model reads **induction ≈ 0.69** (head L10H10) — a real induction head. The onset is *after* the dense-early checkpoint window (still ~chance through step 250).

In [ ]:
import os, glob, torch
os.chdir('/Users/neiltripathi/mech-interp')
import sys; sys.path.insert(0, 'scripts')
# importing transformer_vast loads the BPE tokenizer from models/vast/meta.pt and the
# GPT-2-small config (n_layer=12, n_head=12). Model classes are byte-identical to the
# char/bpe runs, so these checkpoints load with no edits.
from transformer_vast import GPTLanguageModel, encode, decode, vocab_size, n_layer, n_head, device
print('vocab', vocab_size, '| layers', n_layer, '| heads', n_head, '| device', device)

In [ ]:
# hook each head's post-softmax attention pattern
cache = {}
def make_hook(L, H):
    def hook(m, i, o): cache[(L, H)] = o.detach()
    return hook

# repeated-random probe: random token ids (from the FULL 8192 vocab) repeated twice
torch.manual_seed(0)
seq_len = 50
rand = torch.randint(0, vocab_size, (seq_len,))
rep  = torch.cat([rand, rand]).unsqueeze(0).to(device)

def score(model, want_head=False):
    """best-head prev-token + induction scores; loops 12x12 heads. Set want_head to also
    return which (L,H) carries the top induction score."""
    global cache; cache = {}
    hs = [model.blocks[L].sa.heads[H].dropout.register_forward_hook(make_hook(L, H))
          for L in range(n_layer) for H in range(n_head)]
    with torch.no_grad():
        model(rep)
    for h in hs: h.remove()
    prev_best = ind_best = 0.0; ind_head = None
    for L in range(n_layer):
        for H in range(n_head):
            w = cache[(L, H)][0]
            prev = sum(w[i, i-1].item()        for i in range(1, 2*seq_len)) / (2*seq_len-1)
            ind  = sum(w[i, i-seq_len+1].item() for i in range(seq_len, 2*seq_len)) / seq_len
            if prev > prev_best: prev_best = prev
            if ind  > ind_best:  ind_best, ind_head = ind, (L, H)
    return (prev_best, ind_best, ind_head) if want_head else (prev_best, ind_best)

# THE headline: the trained (best-val) model. checkpoints were saved on CUDA -> map_location.
model = GPTLanguageModel()
model.load_state_dict(torch.load('models/vast/model.pt', map_location=device))
model.eval().to(device)
pb, ib, ih = score(model, want_head=True)
print(f'best-val model:  prev-token {pb:.3f}   INDUCTION {ib:.3f}   top induction head {ih}')
print(f'baselines: char 0.017 | small-BPE 0.013')

In [ ]:
# trajectory sweep across whatever checkpoints have downloaded. try/except skips any file
# still mid-download (a partial .pt fails to unzip) so the loop doesn't crash.
steps, prev_s, ind_s = [], [], []
for p in sorted(glob.glob('models/vast/ckpt_*.pt')):
    step = int(p.split('_')[-1].split('.')[0])
    try:
        model.load_state_dict(torch.load(p, map_location=device))
    except Exception as e:
        print(f'step {step:5d}:  SKIP (incomplete/corrupt: {type(e).__name__})'); continue
    model.eval()
    pv, iv = score(model)
    steps.append(step); prev_s.append(pv); ind_s.append(iv)
    print(f'step {step:5d}:  prev-token {pv:.3f}   induction {iv:.3f}')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(steps, prev_s, 'o-', label='previous-token (best head)')
plt.plot(steps, ind_s,  's-', label='induction (best head)')
plt.axhline(0.017, ls='--', color='gray', alpha=0.6, label='char baseline (0.017)')
plt.axhline(0.013, ls=':',  color='gray', alpha=0.6, label='small-BPE baseline (0.013)')
plt.xlabel('training step'); plt.ylabel('best-head score')
plt.title('induction_3: circuit formation at GPT-2-small scale (OpenWebText)')
plt.legend()
plt.savefig('induction_vast.png', dpi=150, bbox_inches='tight')
plt.show()

## Find the head

The top induction head on the trained model (L10H10 in the headline run). On a repeated random sequence, an induction head shows a bright **offset diagonal**: each position in the *second* copy attends back to the token that followed its match in the *first* copy (offset `seq_len - 1`).

In [ ]:
# heatmap of the top induction head on the repeated probe. reload best-val weights first
# (the sweep loop above left the last checkpoint loaded).
model.load_state_dict(torch.load('models/vast/model.pt', map_location=device)); model.eval()
_, _, (L, H) = score(model, want_head=True)   # discover the head, don't hardcode

cache.clear()
h = model.blocks[L].sa.heads[H].dropout.register_forward_hook(make_hook(L, H))
with torch.no_grad(): model(rep)
h.remove()

w = cache[(L, H)][0].cpu()   # (100, 100)
plt.figure(figsize=(8, 8))
plt.imshow(w, cmap='viridis')
plt.axvline(seq_len - 0.5, color='w', lw=0.5, alpha=0.4)
plt.axhline(seq_len - 0.5, color='w', lw=0.5, alpha=0.4)
plt.xlabel('giver (key position)'); plt.ylabel('reader (query position)')
plt.title(f'induction head L{L}H{H} on a repeated random sequence')
plt.colorbar(); plt.savefig('induction_head_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()